<a href="https://colab.research.google.com/github/jahnavimidde/Deep_learning/blob/main/improved_pretrained_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices())
# Must show: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout, BatchNormalization
from keras.callbacks import ReduceLROnPlateau, EarlyStopping
from keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

# ── Data Prep ──
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

X_train = X_train.reshape(-1,28,28,1).astype('float32') / 255
X_test  = X_test.reshape(-1,28,28,1).astype('float32') / 255

X_train_r = tf.image.resize(tf.repeat(X_train, 3, axis=-1), (32,32))
X_test_r  = tf.image.resize(tf.repeat(X_test,  3, axis=-1), (32,32))

Y_train = to_categorical(y_train, 10)
Y_test  = to_categorical(y_test,  10)

# ── Model ──
model_alexnet = Sequential([
    Conv2D(32,  (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    BatchNormalization(),
    MaxPool2D(2,2),

    Conv2D(64,  (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPool2D(2,2),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64,  (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPool2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(10,  activation='softmax')
])

model_alexnet.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Train ──
callbacks = [
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True)
]

model_alexnet.fit(
    X_train_r, Y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_test_r, Y_test),
    callbacks=callbacks
)

loss, acc = model_alexnet.evaluate(X_test_r, Y_test)
print("✅ AlexNet GPU Accuracy:", acc)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
938/938 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - accuracy: 0.8001 - loss: 0.5737 - val_accuracy: 0.8775 - val_loss: 0.3462 - learning_rate: 0.0010
Epoch 2/30
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8807 - loss: 0.3428 - val_accuracy: 0.8886 - val_loss: 0.3077 - learning_rate: 0.0010
Epoch 3/30
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.8969 - loss: 0.2908 - val_accuracy: 0.8885 - val_loss: 0.2995 - learning_rate: 0.0010
Epoch 4/30
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9077 - loss: 0.2654 - val_accuracy: 0.8923 - val_loss: 0.2900 - learning_rate: 0.0010
Epoch 5/30
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9161 - loss: 0.2429 - val_accuracy: 0.8720 - val_loss: 0.3677 - learning_rate: 0.0010
Epoch 6/30
938/938 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.9195 - loss: 0.2284 - val_accuracy: 0.9155 - val_loss: 0.2367 - learning_rate: 0.0010
Epoch 7/30
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9240 - loss: 0.2189

In [5]:
# Define the data for ZF-Net
# Resize images to 64x64 and repeat channels to 3
X_train_resized = tf.image.resize(tf.repeat(X_train, 3, axis=-1), (64,64))
X_test_resized  = tf.image.resize(tf.repeat(X_test,  3, axis=-1), (64,64))

# Use the one-hot encoded labels directly from fashion_mnist
Y_train_alexnet = Y_train # Renaming for consistency with the comment in the original code
Y_test_alexnet  = Y_test  # Renaming for consistency with the comment in the original code

model_zf = Sequential([
    # ZF-Net uses larger first filter (7x7) with stride 2
    Conv2D(96, (7,7), strides=2, activation='relu', padding='same', input_shape=(64,64,3)),
    BatchNormalization(),
    MaxPool2D(3, strides=2),

    Conv2D(256, (5,5), strides=2, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPool2D(3, strides=2),

    Conv2D(384, (3,3), activation='relu', padding='same'),
    BatchNormalization(),

    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(10, activation='softmax')
])

model_zf.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True)
]

model_zf.fit(
    X_train_resized, Y_train_alexnet,
    epochs=30,
    batch_size=32,
    validation_data=(X_test_resized, Y_test_alexnet),
    callbacks=callbacks
)

loss, acc = model_zf.evaluate(X_test_resized, Y_test_alexnet)
print("ZF-Net Accuracy:", acc)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 32s 12ms/step - accuracy: 0.7871 - loss: 0.6246 - val_accuracy: 0.8419 - val_loss: 0.4651 - learning_rate: 0.0010
Epoch 2/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.8725 - loss: 0.3757 - val_accuracy: 0.8486 - val_loss: 0.4115 - learning_rate: 0.0010
Epoch 3/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.8891 - loss: 0.3227 - val_accuracy: 0.8729 - val_loss: 0.3334 - learning_rate: 0.0010
Epoch 4/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.9024 - loss: 0.2857 - val_accuracy: 0.8824 - val_loss: 0.3109 - learning_rate: 0.0010
Epoch 5/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.9112 - loss: 0.2616 - val_accuracy: 0.9001 - val_loss: 0.2682 - learning_rate: 0.0010
Epoch 6/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.9167 - loss: 0.2418 - val_accuracy: 0.8808 - val_loss: 0.3181 - learning_rate: 0.0010
Epoch 7/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accura

In [6]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from keras.models import Model
from keras.layers import Dense, Flatten, Dropout, BatchNormalization
from keras.callbacks import ReduceLROnPlateau, EarlyStopping
from keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

# ── Data Prep ──
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

X_train = X_train.reshape(-1,28,28,1).astype('float32') / 255
X_test  = X_test.reshape(-1,28,28,1).astype('float32') / 255

# VGG needs bigger input — use 48x48 (faster than 96x96, better than 32x32)
X_train_vgg = tf.image.resize(tf.repeat(X_train, 3, axis=-1), (48,48))
X_test_vgg  = tf.image.resize(tf.repeat(X_test,  3, axis=-1), (48,48))

Y_train = to_categorical(y_train, 10)
Y_test  = to_categorical(y_test,  10)

# ── Load Pretrained VGG16 ──
base_model = VGG16(
    weights='imagenet',        # pretrained on ImageNet
    include_top=False,         # remove original classifier
    input_shape=(48,48,3)
)

# ── Freeze early layers, unfreeze last block ──
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze block5 only (last 4 conv layers)
for layer in base_model.layers[-4:]:
    layer.trainable = True

# Show which layers are trainable
for layer in base_model.layers:
    print(f"{layer.name:20s} trainable={layer.trainable}")

# ── Custom Classification Head ──
x = Flatten()(base_model.output)
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
output = Dense(10, activation='softmax')(x)

model_vgg = Model(inputs=base_model.input, outputs=output)

model_vgg.summary()

# ── Compile ──
# Lower LR because we're fine-tuning pretrained weights
model_vgg.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ──
callbacks = [
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                      patience=3, verbose=1, min_lr=1e-7),
    EarlyStopping(monitor='val_accuracy', patience=6,
                  restore_best_weights=True, verbose=1)
]

# ── Train ──
history = model_vgg.fit(
    X_train_vgg, Y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_vgg, Y_test),
    callbacks=callbacks
)

# ── Evaluate ──
loss, acc = model_vgg.evaluate(X_test_vgg, Y_test)
print(f"✅ VGGNet Accuracy: {acc:.4f}")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
input_layer_3        trainable=False
block1_conv1         trainable=False
block1_conv2         trainable=False
block1_pool          trainable=False
block2_conv1         trainable=False
block2_conv2         trainable=False
block2_pool          trainable=False
block3_conv1         trainable=False
block3_conv2         trainable=False
block3_conv3         trainable=False
block3_pool          trainable=False
block4_conv1         trainable=False
block4_conv2         trainable=False
block4_conv3         trainable=False
block4_pool          trainable=False
block5_conv1         trainable=True
block5_conv2         trainable=True
block5_conv3         trainable=True
block5_pool          trainable=True


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 48, 48, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 48, 48, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 48, 48, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 6, 6, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 3, 3, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 1, 1, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 15,114,314 (57.66 MB)

 Trainable params: 7,477,514 (28.52 MB)

 Non-trainable params: 7,636,800 (29.13 MB)

Epoch 1/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 54s 47ms/step - accuracy: 0.8271 - loss: 0.5228 - val_accuracy: 0.8838 - val_loss: 0.3417 - learning_rate: 1.0000e-04
Epoch 2/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 65s 37ms/step - accuracy: 0.8910 - loss: 0.3169 - val_accuracy: 0.9041 - val_loss: 0.2661 - learning_rate: 1.0000e-04
Epoch 3/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 35s 37ms/step - accuracy: 0.9072 - loss: 0.2688 - val_accuracy: 0.8734 - val_loss: 0.3715 - learning_rate: 1.0000e-04
Epoch 4/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 34s 36ms/step - accuracy: 0.9172 - loss: 0.2363 - val_accuracy: 0.9081 - val_loss: 0.2616 - learning_rate: 1.0000e-04
Epoch 5/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 42s 37ms/step - accuracy: 0.9242 - loss: 0.2121 - val_accuracy: 0.9088 - val_loss: 0.2644 - learning_rate: 1.0000e-04
Epoch 6/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 34s 37ms/step - accuracy: 0.9312 - loss: 0.1934 - val_accuracy: 0.9200 - val_loss: 0.2286 - learning_rate: 1.0000e-04
Epoch 7/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 34s 37ms/ste